In [1]:
print("""Retail Store Sales - End-to-End Data Analysis
------------------------------------------------------------------------------
Pipeline:
1. Load raw CSV
2. Clean/reconstruct missing values
3. Compute summary statstics
4. Generate charts (matplotlib)
5. Build a self-contained HTML report with embedded charts

Usage: Python retail_sales_analysis.py
Input: set INPUT_CSV below to your file's path
Outputs: retail_sales_cleaned.csv     - cleaned dataset
         charts/*.png                 - individual chart images
         retail_sales_analysis.html   - final report
""")

Retail Store Sales - End-to-End Data Analysis
------------------------------------------------------------------------------
Pipeline:
1. Load raw CSV
2. Clean/reconstruct missing values
3. Compute summary statstics
4. Generate charts (matplotlib)
5. Build a self-contained HTML report with embedded charts

Usage: Python retail_sales_analysis.py
Input: set INPUT_CSV below to your file's path
Outputs: retail_sales_cleaned.csv     - cleaned dataset
         charts/*.png                 - individual chart images
         retail_sales_analysis.html   - final report



In [3]:
import os
import sys
import json
import base64
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [4]:
INPUT_CSV = 'retail_store_sales.csv'
OUTPUT_DIR = '.'
CHARTS_DIR = os.path.join(OUTPUT_DIR, 'charts')
os.makedirs(CHARTS_DIR, exist_ok=True)

In [5]:
plt.rcParams['font.family'] = 'DejaVu Sans'
COLORS = ['#2E5077', '#79A3B1', '#D4A373', '#B85C38', '#6B8F71', '#C1666B', '#4A5859', '#A26769']

In [6]:
REQUIRED_COLUMNS = [
    'Transaction Date', 'Price Per Unit', 'Quantity', 'Total Spent',
    'Category', 'Item', 'Payment Method', 'Location', 'Customer ID',
]

In [26]:
def load_data(path):
    df = pd.read_csv(path)
    date_col_candidates = [c for c in df.columns if 'date' in c.lower()]
    if 'Transaction Date' not in df.columns and date_col_candidates:
        df = df.rename(columns={date_col_candidates[0]: 'Transaction Date'})
    return df   

In [27]:
def check_schema(df):
    missing = [ c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        print("=" * 70)
        print("SCHEMA MISMATCH: this file is missing columns this script needs.")
        print(f"Missing columns : {missing}")
        print(f"Columns found : {df.columns.tolist()}")
        print("=" * 70)
        print("This script was built for the 'Retail Store Sales' dataset schema.")
        print("If your file has different columns names, either:")
        print(" 1) rename your columns to match REQUIRED_COLUMNS above, or")
        print(" 2) share the column list so the script can be adapted to your schema.")
        sys.exit(1)

In [28]:
def fill_row(row):
    """Reconstruct Price / Quantity / Total Spent from each other where possible
    (Total = Price * Quantity)."""
    p, q, t = row['Price Per Unit'], row['Quantity'], row['Total Spent']
    if pd.isna(t) and not pd.isna(p) and not pd.isna(q):
        t = p * q
    if pd.isna(p) and not pd.isna(t) and not pd.isna(q) and q != 0:
        p = t / q
    if pd.isna(q) and not pd.isna(t) and not pd.isna(p) and p != 0:
        q = t / p
    return pd.Series([p, q, t])

In [29]:
def clean_data(df):
    report = {}
    report['n_rows_raw'] = len(df)
    report['n_cols'] = df.shape[1]
    report['missing_before'] = df.isnull().sum().to_dict()
    report['date_min'] = df['Transaction Date'].min()
    report['date_max'] = df['Transaction Date'].max()
 
    df = df.copy()
    df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
 
    # Discount Applied: missing -> assume False (not recorded / not applied)
    if 'Discount Applied' in df.columns:
        df['Discount Applied'] = (
            df['Discount Applied']
            .astype(str).str.strip().str.lower()
            .map({'true': True, 'false': False, 'yes': True, 'no': False})
            .fillna(False)
            .astype(bool)
        )
    else:
        df['Discount Applied'] = False
 
    # Reconstruct Price / Quantity / Total Spent from each other where possible
    df[['Price Per Unit', 'Quantity', 'Total Spent']] = df.apply(fill_row, axis=1)
    report['missing_after_reconstruction'] = (
        df[['Price Per Unit', 'Quantity', 'Total Spent']].isnull().sum().to_dict()
    )
 
    # Drop rows where revenue still can't be determined
    before_drop = len(df)
    df_clean = df.dropna(subset=['Total Spent', 'Price Per Unit', 'Quantity']).copy()
    report['rows_dropped_missing_value'] = before_drop - len(df_clean)
 
    # Item missing but category known -> label Unknown Item, keep row
    df_clean['Item'] = df_clean['Item'].fillna('Unknown Item')
 
    report['rows_missing_date'] = int(df_clean['Transaction Date'].isna().sum())
    report['n_rows_final'] = len(df_clean)
 
    return df_clean, report

In [30]:
def compute_stats(df_clean, report):
    df_clean['YearMonth'] = df_clean['Transaction Date'].dt.to_period('M')
    df_clean['Year'] = df_clean['Transaction Date'].dt.year
 
    report['n_customers'] = df_clean['Customer ID'].nunique()
    report['n_categories'] = df_clean['Category'].nunique()
    report['n_items'] = df_clean['Item'].nunique()
    report['total_revenue'] = float(df_clean['Total Spent'].sum())
    report['avg_transaction_value'] = float(df_clean['Total Spent'].mean())
    report['median_transaction_value'] = float(df_clean['Total Spent'].median())
    report['date_range'] = (
        f"{df_clean['Transaction Date'].min().date()} to {df_clean['Transaction Date'].max().date()}"
    )
 
    cat_rev = df_clean.groupby('Category')['Total Spent'].sum().sort_values(ascending=True)
    pm = df_clean['Payment Method'].value_counts()
    loc = df_clean['Location'].value_counts()
    disc = df_clean.groupby('Discount Applied')['Total Spent'].agg(['mean', 'count'])
    top_cust = df_clean.groupby('Customer ID')['Total Spent'].sum().sort_values(ascending=False).head(10)
 
    report['top_category_by_revenue'] = cat_rev.idxmax()
    report['top_category_revenue'] = float(cat_rev.max())
    report['bottom_category_by_revenue'] = cat_rev.idxmin()
    report['most_common_payment'] = pm.idxmax()
    report['most_common_payment_pct'] = float(pm.max() / pm.sum() * 100)
    report['online_pct'] = float(loc.get('Online', 0) / loc.sum() * 100)
    report['instore_pct'] = float(loc.get('In-store', 0) / loc.sum() * 100)
    report['discount_txn_pct'] = float(df_clean['Discount Applied'].mean() * 100)
    report['avg_value_no_discount'] = float(disc.loc[False, 'mean']) if False in disc.index else 0.0
    report['avg_value_discount'] = float(disc.loc[True, 'mean']) if True in disc.index else 0.0
    report['top_customer_id'] = top_cust.index[0]
    report['top_customer_spend'] = float(top_cust.iloc[0])
    report['avg_customer_spend'] = float(
        df_clean.groupby('Customer ID')['Total Spent'].sum().mean()
    )
    report['avg_price_per_unit'] = float(df_clean['Price Per Unit'].mean())
    report['avg_quantity'] = float(df_clean['Quantity'].mean())
    report['n_years'] = sorted(df_clean['Year'].dropna().unique().tolist())
 
    return df_clean, report, cat_rev, pm, loc, top_cust

In [34]:
def make_charts(df_clean, cat_rev, pm, loc, top_cust):
    # Monthly revenue trend
    monthly = df_clean.groupby('YearMonth')['Total Spent'].sum().sort_index()
    monthly = monthly[monthly.index.notna()]
    fig, ax = plt.subplots(figsize=(11, 5))
    x = monthly.index.astype(str)
    ax.plot(x, monthly.values, color=COLORS[0], linewidth=2, marker='o', markersize=3)
    ax.fill_between(range(len(x)), monthly.values, alpha=0.15, color=COLORS[0])
    ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold', loc='left')
    ax.set_ylabel('Total Revenue ($)')
    step = max(1, len(x) // 12)
    ax.set_xticks(range(0, len(x), step))
    ax.set_xticklabels([x[i] for i in range(0, len(x), step)], rotation=45, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/monthly_trend.png', dpi=150)
    plt.close()
 
    # Revenue by category
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(cat_rev.index, cat_rev.values, color=COLORS[0])
    for i, bar in enumerate(bars):
        bar.set_color(COLORS[i % len(COLORS)])
    ax.set_title('Total Revenue by Category', fontsize=14, fontweight='bold', loc='left')
    ax.set_xlabel('Total Revenue ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/category_revenue.png', dpi=150)
    plt.close()
 
    # Payment method + location
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].pie(pm.values, labels=pm.index, autopct='%1.1f%%', colors=COLORS[:len(pm)],
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}, textprops={'fontsize': 10})
    axes[0].set_title('Payment Method Share', fontsize=12, fontweight='bold')
    axes[1].pie(loc.values, labels=loc.index, autopct='%1.1f%%', colors=[COLORS[2], COLORS[4]],
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}, textprops={'fontsize': 10})
    axes[1].set_title('Online vs In-store Share', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/payment_location.png', dpi=150)
    plt.close()
 
    # Discount impact
    disc = df_clean.groupby('Discount Applied')['Total Spent'].agg(['mean', 'count'])
    labels = ['No Discount', 'Discount Applied']
    vals = [disc.loc[False, 'mean'] if False in disc.index else 0,
            disc.loc[True, 'mean'] if True in disc.index else 0]
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(labels, vals, color=[COLORS[1], COLORS[3]], width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 2, f'${v:,.2f}', ha='center', fontweight='bold')
    ax.set_title('Average Transaction Value: Discount vs No Discount', fontsize=13, fontweight='bold', loc='left')
    ax.set_ylabel('Average Total Spent ($)')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/discount_impact.png', dpi=150)
    plt.close()
 
    # Top 10 customers
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.bar(top_cust.index, top_cust.values, color=COLORS[0])
    ax.set_title('Top 10 Customers by Total Spend', fontsize=14, fontweight='bold', loc='left')
    ax.set_ylabel('Total Spend ($)')
    plt.xticks(rotation=45, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/top_customers.png', dpi=150)
    plt.close()
 
    # Transaction value distribution
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(df_clean['Total Spent'], bins=40, color=COLORS[0], edgecolor='white', alpha=0.85)
    ax.axvline(df_clean['Total Spent'].mean(), color=COLORS[3], linestyle='--', linewidth=2,
               label=f"Mean: ${df_clean['Total Spent'].mean():.2f}")
    ax.axvline(df_clean['Total Spent'].median(), color=COLORS[4], linestyle='--', linewidth=2,
               label=f"Median: ${df_clean['Total Spent'].median():.2f}")
    ax.set_title('Distribution of Transaction Values', fontsize=14, fontweight='bold', loc='left')
    ax.set_xlabel('Total Spent ($)')
    ax.set_ylabel('Number of Transactions')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/transaction_distribution.png', dpi=150)
    plt.close()
 
    # Yearly revenue by category (stacked bar)
    yearly_cat = df_clean.groupby(['Year', 'Category'])['Total Spent'].sum().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(11, 5))
    yearly_cat.plot(kind='bar', stacked=True, ax=ax, color=COLORS[:len(yearly_cat.columns)])
    ax.set_title('Yearly Revenue by Category', fontsize=14, fontweight='bold', loc='left')
    ax.set_ylabel('Total Revenue ($)')
    ax.set_xlabel('Year')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    plt.xticks(rotation=0)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/yearly_category.png', dpi=150)
    plt.savefig(f'{CHARTS_DIR}/monthly_trend.png', dpi=150)
    plt.close()
 
    # Revenue by category
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(cat_rev.index, cat_rev.values, color=COLORS[0])
    for i, bar in enumerate(bars):
        bar.set_color(COLORS[i % len(COLORS)])
    ax.set_title('Total Revenue by Category', fontsize=14, fontweight='bold', loc='left')
    ax.set_xlabel('Total Revenue ($)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/category_revenue.png', dpi=150)
    plt.close()
 
    # Payment method + location
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].pie(pm.values, labels=pm.index, autopct='%1.1f%%', colors=COLORS[:len(pm)],
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}, textprops={'fontsize': 10})
    axes[0].set_title('Payment Method Share', fontsize=12, fontweight='bold')
    axes[1].pie(loc.values, labels=loc.index, autopct='%1.1f%%', colors=[COLORS[2], COLORS[4]],
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}, textprops={'fontsize': 10})
    axes[1].set_title('Online vs In-store Share', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/payment_location.png', dpi=150)
    plt.close()
 
    # Discount impact
    disc = df_clean.groupby('Discount Applied')['Total Spent'].agg(['mean', 'count'])
    labels = ['No Discount', 'Discount Applied']
    vals = [disc.loc[False, 'mean'] if False in disc.index else 0,
            disc.loc[True, 'mean'] if True in disc.index else 0]
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(labels, vals, color=[COLORS[1], COLORS[3]], width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 2, f'${v:,.2f}', ha='center', fontweight='bold')
    ax.set_title('Average Transaction Value: Discount vs No Discount', fontsize=13, fontweight='bold', loc='left')
    ax.set_ylabel('Average Total Spent ($)')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/discount_impact.png', dpi=150)
    plt.close()
 
    # Top 10 customers
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.bar(top_cust.index, top_cust.values, color=COLORS[0])
    ax.set_title('Top 10 Customers by Total Spend', fontsize=14, fontweight='bold', loc='left')
    ax.set_ylabel('Total Spend ($)')
    plt.xticks(rotation=45, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/top_customers.png', dpi=150)
    plt.close()
 
    # Transaction value distribution
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(df_clean['Total Spent'], bins=40, color=COLORS[0], edgecolor='white', alpha=0.85)
    ax.axvline(df_clean['Total Spent'].mean(), color=COLORS[3], linestyle='--', linewidth=2,
               label=f"Mean: ${df_clean['Total Spent'].mean():.2f}")
    ax.axvline(df_clean['Total Spent'].median(), color=COLORS[4], linestyle='--', linewidth=2,
               label=f"Median: ${df_clean['Total Spent'].median():.2f}")
    ax.set_title('Distribution of Transaction Values', fontsize=14, fontweight='bold', loc='left')
    ax.set_xlabel('Total Spent ($)')
    ax.set_ylabel('Number of Transactions')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/transaction_distribution.png', dpi=150)
    plt.close()
 
    # Yearly revenue by category (stacked bar)
    yearly_cat = df_clean.groupby(['Year', 'Category'])['Total Spent'].sum().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(11, 5))
    yearly_cat.plot(kind='bar', stacked=True, ax=ax, color=COLORS[:len(yearly_cat.columns)])
    ax.set_title('Yearly Revenue by Category', fontsize=14, fontweight='bold', loc='left')
    ax.set_ylabel('Total Revenue ($)')
    ax.set_xlabel('Year')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    plt.xticks(rotation=0)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{CHARTS_DIR}/yearly_category.png', dpi=150)
    plt.close()

In [32]:
def img_tag(name, alt):
    with open(f'{CHARTS_DIR}/{name}.png', 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    return f'<img src="data:image/png;base64,{b64}" alt="{alt}" class="chart-img"/>'
 
 
def build_html_report(report, out_path):
    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Retail Store Sales &mdash; End-to-End Data Analysis</title>
<style>
  :root {{
    --navy: #1F3A54; --blue: #2E5077; --slate: #4A5859;
    --sand: #D4A373; --rust: #B85C38; --bg: #FAF9F6;
    --card: #FFFFFF; --border: #E4E0D8;
  }}
  * {{ box-sizing: border-box; }}
  body {{ margin:0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,Arial,sans-serif;
          background:var(--bg); color:#222; line-height:1.6; }}
  .wrap {{ max-width:980px; margin:0 auto; padding:48px 24px 80px; }}
  header {{ background:linear-gradient(135deg,var(--navy),var(--blue)); color:white;
            padding:56px 24px 64px; text-align:center; }}
  header h1 {{ margin:0 0 8px; font-size:2.1rem; font-weight:700; letter-spacing:-0.02em; }}
  header p {{ margin:0; opacity:0.85; font-size:1.05rem; }}
  .kpi-grid {{ display:grid; grid-template-columns:repeat(4,1fr); gap:16px; margin-top:-40px; }}
  .kpi {{ background:var(--card); border:1px solid var(--border); border-radius:12px;
          padding:20px 16px; text-align:center; box-shadow:0 2px 8px rgba(0,0,0,0.04); }}
  .kpi .val {{ font-size:1.5rem; font-weight:700; color:var(--blue); }}
  .kpi .label {{ font-size:0.8rem; color:var(--slate); text-transform:uppercase; letter-spacing:0.04em; margin-top:4px; }}
  section {{ margin-top:56px; }}
  h2 {{ font-size:1.4rem; color:var(--navy); border-bottom:2px solid var(--sand); padding-bottom:10px; margin-bottom:20px; }}
  h3 {{ color:var(--blue); font-size:1.1rem; margin-bottom:8px; }}
  .card {{ background:var(--card); border:1px solid var(--border); border-radius:12px;
           padding:24px; margin-bottom:24px; box-shadow:0 2px 8px rgba(0,0,0,0.03); }}
  .chart-img {{ width:100%; height:auto; border-radius:8px; display:block; margin:8px 0; }}
  table {{ width:100%; border-collapse:collapse; font-size:0.92rem; }}
  th, td {{ text-align:left; padding:8px 10px; border-bottom:1px solid var(--border); }}
  th {{ color:var(--slate); font-weight:600; }}
  .insight-list {{ padding-left:20px; }}
  .insight-list li {{ margin-bottom:10px; }}
  .tag {{ display:inline-block; background:#EFE8DC; color:var(--rust); border-radius:6px;
          padding:2px 8px; font-size:0.75rem; font-weight:600; margin-right:6px; }}
  footer {{ text-align:center; color:var(--slate); font-size:0.85rem; margin-top:64px; }}
  @media (max-width:700px) {{ .kpi-grid {{ grid-template-columns:repeat(2,1fr); }} }}
</style>
</head>
<body>
<header>
  <h1>Retail Store Sales &mdash; End-to-End Analysis</h1>
  <p>{report['n_rows_final']:,} transactions &middot; {report['date_range']} &middot; {report['n_customers']} customers &middot; {report['n_categories']} categories</p>
</header>
<div class="wrap">
  <div class="kpi-grid">
    <div class="kpi"><div class="val">${report['total_revenue']:,.0f}</div><div class="label">Total Revenue</div></div>
    <div class="kpi"><div class="val">${report['avg_transaction_value']:,.2f}</div><div class="label">Avg Transaction</div></div>
    <div class="kpi"><div class="val">{report['n_rows_final']:,}</div><div class="label">Transactions</div></div>
    <div class="kpi"><div class="val">{report['n_customers']}</div><div class="label">Unique Customers</div></div>
  </div>
 
  <section>
    <h2>1. Data Overview &amp; Cleaning</h2>
    <div class="card">
      <p>The raw dataset contained <strong>{report['n_rows_raw']:,} rows</strong> across <strong>{report['n_cols']} columns</strong>,
      spanning <strong>{report['date_min']}</strong> to <strong>{report['date_max']}</strong>.</p>
      <table>
        <tr><th>Field</th><th>Missing (raw)</th><th>Resolution</th></tr>
        <tr><td>Item</td><td>1,213</td><td>Filled as "Unknown Item"</td></tr>
        <tr><td>Price Per Unit</td><td>609</td><td>Reconstructed from Total &divide; Quantity</td></tr>
        <tr><td>Quantity</td><td>604</td><td>Reconstructed from Total &divide; Price</td></tr>
        <tr><td>Total Spent</td><td>604</td><td>Reconstructed from Price &times; Quantity</td></tr>
        <tr><td>Discount Applied</td><td>4,199</td><td>Treated as "No Discount"</td></tr>
      </table>
      <p style="margin-top:16px;"><strong>{report['rows_dropped_missing_value']} rows</strong> had unrecoverable values and were
      dropped, leaving <strong>{report['n_rows_final']:,} clean transactions</strong>.</p>
    </div>
  </section>
 
  <section>
    <h2>2. Revenue Over Time</h2>
    <div class="card">{img_tag('monthly_trend', 'Monthly revenue trend')}</div>
    <div class="card"><h3>Revenue by Year and Category</h3>{img_tag('yearly_category', 'Yearly revenue by category')}</div>
  </section>
 
  <section>
    <h2>3. Category Performance</h2>
    <div class="card">
      <p><strong>{report['top_category_by_revenue']}</strong> is the top revenue category at
      <strong>${report['top_category_revenue']:,.0f}</strong>; <strong>{report['bottom_category_by_revenue']}</strong> is the lowest.</p>
      {img_tag('category_revenue', 'Revenue by category')}
    </div>
  </section>
 
  <section>
    <h2>4. Channel &amp; Payment Behavior</h2>
    <div class="card">
      <p>{report['online_pct']:.1f}% Online vs {report['instore_pct']:.1f}% In-store. Most common payment:
      <strong>{report['most_common_payment']}</strong> ({report['most_common_payment_pct']:.1f}%).</p>
      {img_tag('payment_location', 'Payment method and location split')}
    </div>
  </section>
 
  <section>
    <h2>5. Discount Effectiveness</h2>
    <div class="card">
      <p>{report['discount_txn_pct']:.1f}% of transactions had a discount. Avg value:
      ${report['avg_value_discount']:,.2f} (discount) vs ${report['avg_value_no_discount']:,.2f} (no discount).</p>
      {img_tag('discount_impact', 'Discount impact')}
    </div>
  </section>
 
  <section>
    <h2>6. Customer Insights</h2>
    <div class="card">
      <p>Top customer <strong>{report['top_customer_id']}</strong>: ${report['top_customer_spend']:,.0f} total,
      vs average customer spend of ${report['avg_customer_spend']:,.0f}.</p>
      {img_tag('top_customers', 'Top 10 customers')}
    </div>
  </section>
 
  <section>
    <h2>7. Transaction Value Distribution</h2>
    <div class="card">
      <p>Mean ${report['avg_transaction_value']:,.2f}, median ${report['median_transaction_value']:,.2f}.</p>
      {img_tag('transaction_distribution', 'Transaction value distribution')}
    </div>
  </section>
 
  <section>
    <h2>8. Key Insights &amp; Recommendations</h2>
    <div class="card">
      <ul class="insight-list">
        <li><span class="tag">Category</span><strong>{report['top_category_by_revenue']}</strong> leads revenue &mdash;
        prioritize stock/marketing here; investigate why <strong>{report['bottom_category_by_revenue']}</strong> lags.</li>
        <li><span class="tag">Channel</span>Online/in-store are nearly balanced ({report['online_pct']:.0f}%/{report['instore_pct']:.0f}%) &mdash;
        treat both channels as equally important.</li>
        <li><span class="tag">Discounts</span>Minimal lift from discounting
        (+${report['avg_value_discount']-report['avg_value_no_discount']:,.2f} avg) &mdash; consider more targeted discounting.</li>
        <li><span class="tag">Customers</span>Top customer drives {report['top_customer_spend']/report['total_revenue']*100:.1f}%
        of total revenue &mdash; a retention program for top spenders could protect this.</li>
        <li><span class="tag">Data Quality</span>~{report['rows_dropped_missing_value']/report['n_rows_raw']*100:.1f}% of raw
        records were unrecoverable &mdash; improving point-of-sale data capture would help future analysis.</li>
      </ul>
    </div>
  </section>
 
  <footer>Generated from {os.path.basename(INPUT_CSV)} &middot; {report['n_rows_final']:,} clean transactions analyzed</footer>
</div>
</body>
</html>
"""
    with open(out_path, 'w') as f:
        f.write(html)

In [35]:
def main():
    df = load_data(INPUT_CSV)
    check_schema(df)  # stop early with a clear message if columns don't match
 
    df_clean, report = clean_data(df)
    df_clean, report, cat_rev, pm, loc, top_cust = compute_stats(df_clean, report)
 
    # Save cleaned data
    df_clean.to_csv(os.path.join(OUTPUT_DIR, 'retail_sales_cleaned.csv'), index=False)
 
    # Charts
    make_charts(df_clean, cat_rev, pm, loc, top_cust)
 
    # Report
    build_html_report(report, os.path.join(OUTPUT_DIR, 'retail_sales_analysis.html'))
 
    # Print summary to console
    print(json.dumps(report, indent=2, default=str))
    print("\nDone. Outputs:")
    print(" - retail_sales_cleaned.csv")
    print(" - charts/*.png")
    print(" - retail_sales_analysis.html")
 
 
if __name__ == '__main__':
    main()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11600\2809100816.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


{
  "n_rows_raw": 12575,
  "n_cols": 11,
  "missing_before": {
    "Transaction ID": 0,
    "Customer ID": 0,
    "Category": 0,
    "Item": 1213,
    "Price Per Unit": 609,
    "Quantity": 604,
    "Total Spent": 604,
    "Payment Method": 0,
    "Location": 0,
    "Transaction Date": 0,
    "Discount Applied": 4199
  },
  "date_min": "2022-01-01",
  "date_max": "2025-01-18",
  "missing_after_reconstruction": {
    "Price Per Unit": 0,
    "Quantity": 604,
    "Total Spent": 604
  },
  "rows_dropped_missing_value": 604,
  "rows_missing_date": 0,
  "n_rows_final": 11971,
  "n_customers": 25,
  "n_categories": 8,
  "n_items": 201,
  "total_revenue": 1552071.0,
  "avg_transaction_value": 129.6525770612313,
  "median_transaction_value": 108.5,
  "date_range": "2022-01-01 to 2025-01-18",
  "top_category_by_revenue": "Butchers",
  "top_category_revenue": 208118.0,
  "bottom_category_by_revenue": "Milk Products",
  "most_common_payment": "Cash",
  "most_common_payment_pct": 34.2744967003592,

In [37]:
from IPython.display import IFrame
IFrame('retail_sales_analysis.html', width=1000, height=800)